<a href="https://colab.research.google.com/github/abdullahmujahidali/AFM-IMS/blob/main/VetAI_Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langgraph langchain_openai openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.7/412.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.3 MB/s eta 0:00:00


In [ ]:
import os
from typing import List, Dict, Tuple, Annotated, TypedDict, Union, Any
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
import json

from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

In [12]:
class AgentState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]]
    current_input: str
    results: Dict[str, Any]


def create_agent(name: str):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are {name}, an AI agent responsible for processing user input. "
                  "Analyze the input and provide your assessment."),
        ("human", "{input}")
    ])

    def agent_function(state: AgentState):
        current_input = state["current_input"]

        messages = prompt.format_messages(
            name=name,
            input=current_input
        )

        response = llm.invoke(messages)

        new_state = state.copy()
        new_state["messages"].append(response)
        new_state["results"][name] = response.content

        return new_state

    return agent_function

first_agent = create_agent("DataAnalyzer")


In [13]:
def create_summarizer_agent():
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a Summarizer agent. Your job is to take the analysis from the DataAnalyzer "
                  "and create a brief, clear summary."),
        ("human", "Please summarize this analysis: {analysis}")
    ])

    def summarizer_function(state: AgentState):
        analysis = state["results"]["DataAnalyzer"]

        messages = prompt.format_messages(
            analysis=analysis
        )
        response = llm.invoke(messages)

        new_state = state.copy()
        new_state["messages"].append(response)
        new_state["results"]["Summarizer"] = response.content

        return new_state

    return summarizer_function


In [14]:
summarizer_agent = create_summarizer_agent()

workflow = StateGraph(AgentState)

workflow.add_node("analyze", first_agent)
workflow.add_node("summarize", summarizer_agent)

workflow.set_entry_point("analyze")

workflow.add_edge("analyze", "summarize")
workflow.add_edge("summarize", END)

app = workflow.compile()

initial_state = {
    "messages": [],
    "current_input": "This is a test input that needs to be analyzed.",
    "results": {}
}

result = app.invoke(initial_state)
print("\nAnalysis:", result["results"]["DataAnalyzer"])
print("\nSummary:", result["results"]["Summarizer"])


Analysis: Thank you for providing the input. As an AI DataAnalyzer, I can see that the input is a simple text string indicating that it is a test input that requires analysis. If you have any specific questions or tasks related to this input, please let me know how I can assist you further.

Summary: The analysis indicates that the input is a text string for testing that requires further analysis. If there are specific questions or tasks related to this input, the AI DataAnalyzer is ready to assist.
